In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

In [2]:
tracks = pd.read_csv('spotify-tracks-dataset.csv')
user = pd.read_csv('spotify_user_behavior_realistic_50000_rows.csv')
top50 = pd.read_csv('spotify-streaming-top-50-world.csv')
spain = pd.read_csv('spotify-streaming-top-50-spain.csv')

In [3]:
dataframes = {
    "Tracks Dataset": tracks,
    "User Behavior": user,
    "Top 50 World": top50,
    "Top 50 Spain": spain
}

for name, df in dataframes.items():
    print(f"--- {name} Columns ---")
    print(list(df.columns))
    print("\n" + "="*30 + "\n")

--- Tracks Dataset Columns ---
['Unnamed: 0.1', 'Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


--- User Behavior Columns ---
['user_id', 'country', 'age', 'signup_date', 'subscription_type', 'subscription_status', 'months_inactive', 'inactive_3_months_flag', 'ad_interaction', 'ad_conversion_to_subscription', 'music_suggestion_rating_1_to_5', 'avg_listening_hours_per_week', 'favorite_genre', 'most_liked_feature', 'desired_future_feature', 'primary_device', 'playlists_created', 'avg_skips_per_day']


--- Top 50 World Columns ---
['date', 'position', 'song', 'artist', 'popularity', 'duration_ms', 'album_type', 'total_tracks', 'release_date', 'is_explicit', 'album_cover_url']


--- Top 50 Spain Columns ---
['date', 'position', 'song', 'artist', 'popularity', 'dura

In [5]:
tracks.info()
user.info()
top50.info()
spain.info()

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0.1      114000 non-null  int64  
 1   Unnamed: 0        114000 non-null  int64  
 2   track_id          114000 non-null  str    
 3   artists           113999 non-null  str    
 4   album_name        113999 non-null  str    
 5   track_name        113999 non-null  str    
 6   popularity        114000 non-null  int64  
 7   duration_ms       114000 non-null  int64  
 8   explicit          114000 non-null  bool   
 9   danceability      114000 non-null  float64
 10  energy            114000 non-null  float64
 11  key               114000 non-null  int64  
 12  loudness          114000 non-null  float64
 13  mode              114000 non-null  int64  
 14  speechiness       114000 non-null  float64
 15  acousticness      114000 non-null  float64
 16  instrumentalness  114000 non-nu

## Καθαρισμός δεδομένων (Data cleaning)

In [25]:
tracks.isnull().sum()

track_id            0
artists             0
album_name          0
track_name          0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

In [26]:
tracks = tracks.dropna()

print(tracks.isnull().sum())

track_id            0
artists             0
album_name          0
track_name          0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64


In [27]:
user.isnull().sum()

user_id                           0
country                           0
age                               0
signup_date                       0
subscription_type                 0
subscription_status               0
months_inactive                   0
inactive_3_months_flag            0
ad_interaction                    0
ad_conversion_to_subscription     0
music_suggestion_rating_1_to_5    0
avg_listening_hours_per_week      0
favorite_genre                    0
most_liked_feature                0
desired_future_feature            0
primary_device                    0
playlists_created                 0
avg_skips_per_day                 0
dtype: int64

In [28]:
top50.isnull().sum()

date               0
position           0
song               0
artist             0
popularity         0
duration_ms        0
album_type         0
total_tracks       0
release_date       0
is_explicit        0
album_cover_url    0
dtype: int64

In [29]:
spain.isnull().sum()

date               0
position           0
song               0
artist             0
popularity         0
duration_ms        0
album_type         0
total_tracks       0
release_date       0
is_explicit        0
album_cover_url    0
dtype: int64

In [30]:
# Έλεγχος για διπλότυπα track_id
duplicates_count = tracks.duplicated(subset=['track_id']).sum()
print(f"Βρέθηκαν {duplicates_count} διπλότυπα tracks.")

# Διαγραφή διπλότυπων (κρατάμε την πρώτη εμφάνιση)
tracks = tracks.drop_duplicates(subset=['track_id'], keep='first')


Βρέθηκαν 0 διπλότυπα tracks.


In [31]:
# Αφαίρεση στηλών που ξεκινάνε με Unnamed
tracks = tracks.loc[:, ~tracks.columns.str.contains('^Unnamed')]

In [32]:
# Δες τα στατιστικά για να εντοπίσεις ακραίες τιμές (π.χ. min duration 0)
print(tracks[['duration_ms', 'popularity', 'danceability']].describe())

# Παράδειγμα: Κράτα μόνο τραγούδια με διάρκεια πάνω από 10 δευτερόλεπτα
tracks = tracks[tracks['duration_ms'] > 10000]

        duration_ms    popularity  danceability
count  8.973900e+04  89739.000000  89739.000000
mean   2.291468e+05     33.199178      0.562173
std    1.129440e+05     20.580457      0.176683
min    1.338600e+04      0.000000      0.000000
25%    1.730400e+05     19.000000      0.450000
50%    2.132980e+05     33.000000      0.576000
75%    2.642930e+05     49.000000      0.692000
max    5.237295e+06    100.000000      0.985000


In [33]:

tracks = tracks.reset_index(drop=True)

In [34]:
user = user.drop_duplicates(subset=['user_id'])

In [35]:
print(user['age'].describe())

count    50000.000000
mean        38.010280
std         12.984989
min         16.000000
25%         27.000000
50%         38.000000
75%         49.000000
max         60.000000
Name: age, dtype: float64
